
# Learning the tree: TAN and naive Bayes

:class:`skbn.TAN` (tree augmented naive Bayes) lets every attribute depend
on the class and on at most one other attribute. The dependencies form a
tree, learned from the data with the Chow-Liu procedure: the conditional
mutual information $I(A_i; A_j \mid C)$ is computed for every pair of
attributes, and the tree of maximum total weight is kept.

The data have seven categorical attributes. Attributes 0-1-2 and 3-4-5 form
two chains: each attribute copies the previous one shifted by the class,
with some noise. Attribute 6 is pure noise. Every attribute on its own is
uniformly distributed in both classes, so it carries no information about
the class; the information is only in the pairs of consecutive attributes.

Naive Bayes looks at one attribute at a time and stays at chance level.
TAN finds the two chains (the edges of largest conditional mutual
information), joins them and the noise attribute with weak edges, and
classifies almost perfectly with only a few dozen training instances.


In [ ]:
# Author: The scikit-bayes Developers
# SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import NullLocator, StrMethodFormatter
from sklearn.metrics import accuracy_score

from skbn import TAN, MixedNB

# Two chains and a noise attribute
N_VALUES = 4


def make_data(n_samples, rng):
    y = rng.randint(0, 2, size=n_samples)
    columns = []
    for keep in (0.9, 0.75):  # Two chains with different noise levels
        chain = [rng.randint(0, N_VALUES, size=n_samples)]
        for _ in range(2):
            copy = (chain[-1] + y) % N_VALUES
            noise = rng.randint(0, N_VALUES, size=n_samples)
            chain.append(np.where(rng.rand(n_samples) < keep, copy, noise))
        columns.extend(chain)
    columns.append(rng.randint(0, N_VALUES, size=n_samples))  # Noise
    return np.column_stack(columns), y


rng = np.random.RandomState(42)
X_train, y_train = make_data(2000, rng)
X_test, y_test = make_data(5000, rng)

tan = TAN().fit(X_train, y_train)
cmi = tan.conditional_mutual_info_
print("Parent of each attribute:", tan.parents_)

# Accuracy as the training set grows
sizes = [25, 50, 100, 200, 500, 1000, 2000]
models = {
    "MixedNB": (MixedNB, "#6E6E6E", "^"),
    "TAN": (TAN, "#E69F00", "o"),
}
accuracy = {name: [] for name in models}
for n in sizes:
    for name, (Model, _, _) in models.items():
        model = Model().fit(X_train[:n], y_train[:n])
        accuracy[name].append(accuracy_score(y_test, model.predict(X_test)))

for name in models:
    print(f"{name}: {np.round(accuracy[name], 3)}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
n_features = X_train.shape[1]

# Conditional mutual information, with the edges of the tree framed
im = axes[0].imshow(cmi, cmap="viridis")
for i, p in enumerate(tan.parents_):
    if p >= 0:
        for r, c in ((i, p), (p, i)):
            axes[0].add_patch(
                plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, color="white", lw=2)
            )
axes[0].set_xticks(range(n_features))
axes[0].set_yticks(range(n_features))
axes[0].set_xlabel("Attribute $A_j$")
axes[0].set_ylabel("Attribute $A_i$")
axes[0].grid(False)
axes[0].set_title("Conditional mutual information\nTree edges framed")
fig.colorbar(im, ax=axes[0], shrink=0.8, label=r"$I(A_i; A_j \mid C)$ (nats)")

# The learned tree, directed away from the root (attribute 0)
angles = np.pi / 2 - 2 * np.pi * np.arange(n_features) / n_features
pos = np.column_stack([np.cos(angles), np.sin(angles)])
max_cmi = max(cmi[i, p] for i, p in enumerate(tan.parents_) if p >= 0)
for i, p in enumerate(tan.parents_):
    if p < 0:
        continue
    axes[1].annotate(
        "",
        xy=pos[i],
        xytext=pos[p],
        arrowprops=dict(
            arrowstyle="-|>",
            color="#E69F00",
            lw=1 + 4 * cmi[i, p] / max_cmi,
            shrinkA=13,
            shrinkB=13,
        ),
    )
    # Label outside the circle for the edges between neighbours, on the
    # middle of the edge for the chords
    mid = (pos[i] + pos[p]) / 2
    if np.linalg.norm(mid) > 0.75:
        mid = mid * (1 + 0.17 / np.linalg.norm(mid))
    axes[1].text(
        *mid,
        f"{cmi[i, p]:.2f}",
        ha="center",
        va="center",
        bbox=dict(facecolor="white", edgecolor="none", pad=1),
    )
# Chain attributes in dark grey, the noise attribute in light grey
node_colors = ["#1A1A1A"] * 6 + ["#999999"]
for i in range(n_features):
    axes[1].scatter(*pos[i], s=500, color=node_colors[i], zorder=3)
    axes[1].text(*pos[i], str(i), color="white", ha="center", va="center", zorder=4)
axes[1].set_xlim(-1.3, 1.3)
axes[1].set_ylim(-1.3, 1.3)
axes[1].set_aspect("equal")
axes[1].axis("off")
axes[1].set_title("Tree learned by TAN\nEdges labelled with $I(A_i; A_j \\mid C)$")

for name, (_, color, marker) in models.items():
    axes[2].plot(sizes, accuracy[name], color=color, marker=marker, label=name)
axes[2].axhline(0.5, color="#999999", ls=":", label="Chance")
axes[2].set_xscale("log")
axes[2].set_xticks(sizes)
axes[2].xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
axes[2].xaxis.set_minor_locator(NullLocator())
axes[2].set_xlabel("Training set size")
axes[2].set_ylabel("Test accuracy")
axes[2].set_title("Test accuracy")
axes[2].legend(loc="center right")

fig.suptitle("TAN recovers the dependencies that naive Bayes ignores")
plt.show()